<a href="https://colab.research.google.com/github/Chimatanagagopal/RAG/blob/main/graphRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install neo4j

In [16]:
from neo4j import GraphDatabase

URI = "neo4j+s://51612bdc.databases.neo4j.io"
USERNAME = "51612bdc"
PASSWORD = "2TciH7DRV-9AAbq_yaPyJBGum1aLMbBgsen9VxKP6lM"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j!")

Connected to Neo4j!


In [17]:



def create_nodes(tx):
    tx.run("""
        CREATE (python:Technology {name: "Python"})
        CREATE (ml:Technology {name: "Machine Learning"})
        CREATE (numpy:Library {name: "NumPy"})
        CREATE (pandas:Library {name: "Pandas"})
    """)


def create_relationships(tx):
    tx.run("""
        MATCH (python:Technology {name: "Python"})
        MATCH (ml:Technology {name: "Machine Learning"})
        MATCH (numpy:Library {name: "NumPy"})
        MATCH (pandas:Library {name: "Pandas"})

        CREATE (python)-[:USED_FOR]->(ml)
        CREATE (ml)-[:USES]->(numpy)
        CREATE (ml)-[:USES]->(pandas)
    """)


with driver.session() as session:
    session.execute_write(create_nodes)
    session.execute_write(create_relationships)

print("Graph created!")

driver.close()

Graph created!


In [18]:
from neo4j import GraphDatabase

URI = "neo4j+s://51612bdc.databases.neo4j.io"
USERNAME = "51612bdc"
PASSWORD = "2TciH7DRV-9AAbq_yaPyJBGum1aLMbBgsen9VxKP6lM"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected!")

Connected!


In [22]:
def get_libraries(tx):
    result = tx.run("""
        MATCH (ml:Technology {name: "Machine Learning"})
              -[:USES]->
              (library)
        RETURN library.name AS library
    """)

    return [record["library"] for record in result]


with driver.session() as session:
    libraries = session.execute_read(get_libraries)

print("Libraries used by Machine Learning:")

for library in libraries:
    print(library)

Libraries used by Machine Learning:
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumP

In [23]:
def get_related_nodes(tx, source_name):
    result = tx.run("""
        MATCH (source {name: $source_name})
              -[r]->
              (target)
        RETURN type(r) AS relationship, target.name AS target
    """, source_name=source_name)

    return [
        {
            "relationship": record["relationship"],
            "target": record["target"]
        }
        for record in result
    ]


source = "Machine Learning"

with driver.session() as session:
    results = session.execute_read(
        get_related_nodes,
        source
    )

print(results)

[{'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, 

In [24]:
!pip install -q transformers accelerate sentence-transformers neo4j

In [25]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.8 MB/s eta 0:00:00


In [26]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model loaded!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!


In [28]:
def extract_relationships(text):
    prompt = f"""
You are a knowledge graph extraction system.

Extract entities and relationships from the following text.

Text:
{text}

Return ONLY in this format:

ENTITY: entity_name
ENTITY: entity_name
RELATIONSHIP: source | relationship | target
"""

    messages = [
        {
            "role": "system",
            "content": "You extract entities and relationships for a knowledge graph."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=True
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [29]:
text = """
Python is widely used for Machine Learning.
Machine Learning projects commonly use NumPy and Pandas.
Pandas is a Python library used for data analysis.
"""

result = extract_relationships(text)

print(result)

ENTITY: Python
ENTITY: Machine Learning
ENTITY: NumPy
ENTITY: Pandas
RELATIONSHIP: is used for | by
RELATIONSHIP: is used in | by
RELATIONSHIP: is used with | by
